# RL Pong training notebook

This notebook trains a pong policy against a mostly-tracking bot. The bot follows the ball closely but occasionally makes mistakes, which gives the network a fair learning target.

In [1]:
import numpy as np
import pygame
from enum import Enum
import pong_engine

np.set_printoptions(precision=3, suppress=True)


pygame 2.6.1 (SDL 2.32.10, Python 3.14.4)
Hello from the pygame community. https://www.pygame.org/contribute.html


<frozen importlib._bootstrap>:491: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.


In [2]:
# Loss functions used by the policy and diagnostic code.
class LossFunction(Enum):
    MSE = 1
    BCE = 2
    CCE = 3
    REINFORCE = 4

# Weight initialization strategies.
class WeightInitialization(Enum):
    Random = 1
    Xavier = 2
    He = 3

# Base layer interface.
class Layer:
    def __init__(self):
        pass

    def forward_pass(self, x):
        return x

    def backward_pass(self, derivative):
        return derivative

    def update(self, derivative):
        pass

class Sigmoid(Layer):
    def __init__(self):
        super().__init__()
        self.sigmoid_output = None

    def forward_pass(self, x):
        self.sigmoid_output = 1.0 / (1.0 + np.exp(-x))
        return self.sigmoid_output

    def backward_pass(self, derivative):
        return derivative * self.sigmoid_output * (1.0 - self.sigmoid_output)

class Relu(Layer):
    def __init__(self):
        super().__init__()
        self.x = None
        self.gradient = None

    def forward_pass(self, x):
        self.x = x
        self.gradient = np.where(x > 0, 1.0, 0.0)
        return self.x * self.gradient

    def backward_pass(self, derivative):
        return derivative * self.gradient

class Softmax(Layer):
    def __init__(self):
        super().__init__()
        self.probabilities = None

    def forward_pass(self, x):
        shifted = x - np.max(x, axis=0, keepdims=True)
        temp = np.exp(shifted)
        denominator = np.sum(temp, axis=0, keepdims=True)
        denominator = np.clip(denominator, 1e-12, None)
        self.probabilities = temp / denominator
        self.probabilities = np.clip(self.probabilities, 1e-8, 1.0 - 1e-8)
        self.probabilities /= np.sum(self.probabilities, axis=0, keepdims=True)
        return self.probabilities

    def backward_pass(self, derivative):
        sum_term = np.sum(derivative * self.probabilities, axis=0, keepdims=True)
        return self.probabilities * (derivative - sum_term)

    def update(self, derivative):
        return

class Linear(Layer):
    def __init__(self, input_dim, output_dim, learning_rate=0.001, weight_init=WeightInitialization.Random, weights=None, biases=None):
        super().__init__()
        if weights is None:
            match weight_init:
                case WeightInitialization.Random:
                    weights = np.random.uniform(-1.0, 1.0, size=(output_dim, input_dim))
                case WeightInitialization.Xavier:
                    limit = np.sqrt(6.0 / (input_dim + output_dim))
                    weights = np.random.uniform(-limit, limit, size=(output_dim, input_dim))
                case WeightInitialization.He:
                    weights = np.random.normal(0.0, np.sqrt(2.0 / input_dim), size=(output_dim, input_dim))
        if biases is None:
            biases = np.full((output_dim, 1), 0.0, dtype=float)
        self.learning_rate = learning_rate
        self.x = None
        self.weights = weights
        self.biases = biases

    def forward_pass(self, x):
        self.x = x
        return self.weights @ x + self.biases

    def backward_pass(self, derivative):
        return self.weights.T @ derivative

    def update(self, derivative):
        weight_gradient = derivative @ self.x.T
        bias_gradient = np.mean(derivative, axis=1, keepdims=True)
        self.weights -= self.learning_rate * weight_gradient
        self.biases -= self.learning_rate * bias_gradient

class Model:
    def __init__(self, layers):
        self.layers = layers

    def calc_loss(self, y_hat, y, loss_function=LossFunction.MSE, reinforce_actions=None):
        match loss_function:
            case LossFunction.MSE:
                return y_hat - y
            case LossFunction.BCE:
                return (y_hat - y) / (y_hat * (1.0 - y_hat))
            case LossFunction.CCE:
                return y * np.log(np.clip(y_hat, 1e-8, 1.0))
            case LossFunction.REINFORCE:
                if reinforce_actions is None:
                    indexes = np.argmax(y_hat, axis=0)
                    reinforce_actions = np.eye(y_hat.shape[0], dtype=float)[indexes].T
                safe_y_hat = np.clip(y_hat, 1e-8, 1.0)
                return -y.reshape(1, -1) * reinforce_actions / safe_y_hat

    def calc_cost(self, y_hat, y, loss_function=LossFunction.MSE, reinforce_actions=None):
        match loss_function:
            case LossFunction.MSE:
                return np.mean((y_hat - y) ** 2) / 2.0
            case LossFunction.BCE:
                return -np.mean(y * np.log(np.clip(y_hat, 1e-8, 1.0)) + (1.0 - y) * np.log(np.clip(1.0 - y_hat, 1e-8, 1.0)))
            case LossFunction.CCE:
                return -np.mean(np.sum(y * np.log(np.clip(y_hat, 1e-8, 1.0)), axis=0, keepdims=True))
            case LossFunction.REINFORCE:
                return np.mean(self.calc_loss(y_hat, y, loss_function, reinforce_actions=reinforce_actions))

    def forward_pass(self, x):
        inp = x
        for layer in self.layers:
            inp = layer.forward_pass(inp)
        return inp

    def backward_pass(self, derivative):
        gradient = derivative
        for layer in reversed(self.layers):
            temp_gradient = layer.backward_pass(gradient)
            layer.update(gradient)
            gradient = temp_gradient

# Model hyperparameters for the Pong policy.
LEARNING_RATE = 0.01
EXPLORATION_EPSILON = 0.10
REWARD_DISCOUNT = 0.99
NUM_ACTIONS = 3

model = Model([
    Linear(8, 16, LEARNING_RATE, WeightInitialization.He),
    Relu(),
    Linear(16, 16, LEARNING_RATE, WeightInitialization.He),
    Relu(),
    Linear(16, 3, LEARNING_RATE, WeightInitialization.He),
    Softmax(),
])


In [3]:
# Pong game configuration and engine initialization.
# This creates the actual pong environment used in training and in the final play cell.
FIELD_WIDTH = 640
FIELD_HEIGHT = 480
BALL_RADIUS = 10
PLAYER_WIDTH = 10
PLAYER_HEIGHT = 100
PLAYER_OFFSET = 80
PLAYER_ACCEL = 2000.0
PLAYER_TOP_SPEED = 500.0
BALL_ACCEL = 1.05
BALL_VEL = 170.0

game = pong_engine.Game(
    ball_radius=BALL_RADIUS,
    player_width=PLAYER_WIDTH,
    player_height=PLAYER_HEIGHT,
    field_width=FIELD_WIDTH,
    field_height=FIELD_HEIGHT,
    player_offset=PLAYER_OFFSET,
    player_accel=PLAYER_ACCEL,
    player_top_speed=PLAYER_TOP_SPEED,
    ball_accel=BALL_ACCEL,
    ball_vel=BALL_VEL,
)

ACTION_MAP = {
    0: pong_engine.Move.UP,
    1: pong_engine.Move.NONE,
    2: pong_engine.Move.DOWN,
}


In [4]:
# Convert the raw game state into a normalized 8-element numeric vector.
# Inputs are: [field width, field height, ball x, ball y, player 1 x, player 1 y, player 2 x, player 2 y]
# Values are scaled so the policy network sees stable ranges instead of large raw game coordinates.
def state_to_np(game_state, field_width, field_height, normalization_factor=100.0):
    return np.array([
        float(field_width) / normalization_factor,
        float(field_height) / normalization_factor,
        float(game_state.ball_x) / normalization_factor * 10.0,
        float(game_state.ball_y) / normalization_factor * 10.0,
        float(game_state.player_one_x) / normalization_factor,
        float(game_state.player_one_y) / normalization_factor,
        float(game_state.player_two_x) / normalization_factor,
        float(game_state.player_two_y) / normalization_factor,
    ], dtype=float)


In [5]:
# Pick the model's next action from the current state.
# The network outputs 3 action scores (UP, NONE, DOWN), then we choose the argmax.
# A small epsilon keeps exploration active during early training.
def choose_policy_action(model, game_state):
    state_vector = state_to_np(game_state, FIELD_WIDTH, FIELD_HEIGHT, 100.0).reshape(-1, 1)
    probabilities = model.forward_pass(state_vector)
    if np.random.rand() < EXPLORATION_EPSILON:
        action_index = int(np.random.randint(0, NUM_ACTIONS))
    else:
        action_index = int(np.argmax(probabilities, axis=0)[0])
    return action_index, probabilities

# Map action index back to the pong_engine enum value.
def action_from_index(action_index):
    return ACTION_MAP.get(action_index, pong_engine.Move.NONE)


In [6]:
# Player 2 bot: mostly mimic the ball's Y position while allowing some mistakes.
# This is intentionally a strong but not perfect bot so the RL agent can still improve.
def bot_move_for_player_two(game_state, deadzone=10.0, noise_prob=0.08):
    paddle_center = game_state.player_two_y + PLAYER_HEIGHT / 2.0
    error = game_state.ball_y - paddle_center

    if abs(error) <= deadzone:
        return pong_engine.Move.NONE

    if np.random.rand() < noise_prob:
        return pong_engine.Move.NONE if np.random.rand() < 0.5 else (pong_engine.Move.UP if error < 0 else pong_engine.Move.DOWN)

    return pong_engine.Move.UP if error < 0 else pong_engine.Move.DOWN

# Keyboard helper for human-controlled player 2.
def get_move_from_keys(up_key, down_key):
    if up_key and not down_key:
        return pong_engine.Move.UP
    if down_key and not up_key:
        return pong_engine.Move.DOWN
    return pong_engine.Move.NONE


In [7]:
# Run a single Pong episode using the current model for player 1 and the bot for player 2.
# This returns the state history and reward history needed for REINFORCE training.
def run_episode(model, render=False, human_player_two=False, max_steps=2000):
    states = []
    action_probs = []
    actions = []
    rewards = []

    if render:
        pygame.init()
        clock = pygame.time.Clock()
        screen = pygame.display.set_mode((FIELD_WIDTH, FIELD_HEIGHT))
        pygame.display.set_caption('Pong Training Episode')

    last_state = game.step(pong_engine.Move.NONE, pong_engine.Move.NONE, 1.0 / 60.0)
    for _ in range(max_steps):
        if render:
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    return states, actions, action_probs, rewards, 0

        state_vector = state_to_np(last_state, FIELD_WIDTH, FIELD_HEIGHT, 100.0)
        action_index, action_prob = choose_policy_action(model, last_state)
        player_one_move = action_from_index(action_index)

        if human_player_two:
            pressed = pygame.key.get_pressed()
            player_two_move = get_move_from_keys(pressed[pygame.K_UP], pressed[pygame.K_DOWN])
        else:
            player_two_move = bot_move_for_player_two(last_state)

        states.append(state_vector.copy())
        actions.append(action_index)
        action_probs.append(action_prob.copy())

        delta_time = 1.0 / 60.0
        result = game.step(player_one_move, player_two_move, delta_time)

        # Reward is based on staying near the ball while also rewarding wins and penalizing losses.
        reward = -abs((result.player_one_y + PLAYER_HEIGHT / 2.0) - result.ball_y) / FIELD_HEIGHT
        if result.winner == 1:
            reward += 5.0
        elif result.winner == 2:
            reward -= 5.0
        rewards.append(reward)
        last_state = result

        if render:
            screen.fill((0, 0, 0))
            pygame.draw.rect(screen, (255, 255, 255), pygame.Rect(0, 0, FIELD_WIDTH, FIELD_HEIGHT), 2)
            pygame.draw.line(screen, (255, 255, 255), (FIELD_WIDTH // 2, 0), (FIELD_WIDTH // 2, FIELD_HEIGHT), 2)
            left_paddle = pygame.Rect(int(result.player_one_x), int(result.player_one_y), PLAYER_WIDTH, PLAYER_HEIGHT)
            right_paddle = pygame.Rect(int(result.player_two_x), int(result.player_two_y), PLAYER_WIDTH, PLAYER_HEIGHT)
            pygame.draw.rect(screen, (255, 255, 255), left_paddle)
            pygame.draw.rect(screen, (255, 255, 255), right_paddle)
            pygame.draw.circle(screen, (255, 255, 255), (int(result.ball_x), int(result.ball_y)), BALL_RADIUS)
            pygame.display.flip()
            pygame.time.delay(16)

        if result.game_over:
            break

    if render:
        pygame.quit()
    winner = 0 if len(rewards) == 0 else (1 if last_state.winner == 1 else 2 if last_state.winner == 2 else 0)
    return states, actions, action_probs, rewards, winner


In [8]:
# Discount rewards backwards so earlier actions are credited for later outcomes.
# This helps the policy learn which decisions led to a good or bad ending.
def discount_rewards(rewards, gamma=REWARD_DISCOUNT):
    discounted = np.zeros_like(rewards, dtype=float)
    running = 0.0
    for i in range(len(rewards) - 1, -1, -1):
        running = rewards[i] + gamma * running
        discounted[i] = running
    discounted = (discounted - np.mean(discounted)) / (np.std(discounted) + 1e-8)
    return discounted


In [ ]:
# Training loop configuration.
# num_iterations: total episodes to run.
# training_log_every: how often to print summary stats.
num_iterations = 100000
training_log_every = 1000
wins = 0
last_returns = []

for iteration in range(1, num_iterations + 1):
    states, actions, prob_hist, step_rewards, winner = run_episode(model, render=False, human_player_two=False)
    if len(step_rewards) == 0:
        continue

    returns = discount_rewards(np.asarray(step_rewards, dtype=float))
    last_returns.append(np.mean(returns))
    wins += int(winner == 1)

    state_batch = np.array(states).T
    policy_output = model.forward_pass(state_batch)
    action_one_hot = np.eye(NUM_ACTIONS, dtype=float)[actions].T
    loss = model.calc_loss(policy_output, returns, LossFunction.REINFORCE, reinforce_actions=action_one_hot)
    model.backward_pass(np.clip(loss, -5.0, 5.0))

    if iteration % training_log_every == 0:
        recent_win_rate = wins / max(1, iteration)
        print(f'iteration={iteration} avg_return={np.mean(last_returns[-training_log_every:]):.3f} win_rate={recent_win_rate:.3f}')


## Play against the trained model

Run this cell to play against the model. The model controls player 1 and keyboard controls player 2.

In [10]:
# Human-vs-model play mode.
# Player 2 is controlled with the keyboard; player 1 is your trained policy.
pygame.init()
screen = pygame.display.set_mode((FIELD_WIDTH, FIELD_HEIGHT))
pygame.display.set_caption('RL Pong Match')
clock = pygame.time.Clock()
running = True
last_state = game.step(pong_engine.Move.NONE, pong_engine.Move.NONE, 1.0 / 60.0)

while running:
    delta_time = 1.0 / 60.0
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
    if not running:
        break

    pressed = pygame.key.get_pressed()
    right_move = get_move_from_keys(pressed[pygame.K_UP], pressed[pygame.K_DOWN])
    action_index, _ = choose_policy_action(model, last_state)
    left_move = action_from_index(action_index)
    result = game.step(left_move, right_move, delta_time)
    last_state = result

    screen.fill((0, 0, 0))
    pygame.draw.rect(screen, (255, 255, 255), pygame.Rect(0, 0, FIELD_WIDTH, FIELD_HEIGHT), 2)
    pygame.draw.line(screen, (255, 255, 255), (FIELD_WIDTH // 2, 0), (FIELD_WIDTH // 2, FIELD_HEIGHT), 2)
    left_paddle = pygame.Rect(int(result.player_one_x), int(result.player_one_y), PLAYER_WIDTH, PLAYER_HEIGHT)
    right_paddle = pygame.Rect(int(result.player_two_x), int(result.player_two_y), PLAYER_WIDTH, PLAYER_HEIGHT)
    pygame.draw.rect(screen, (255, 255, 255), left_paddle)
    pygame.draw.rect(screen, (255, 255, 255), right_paddle)
    pygame.draw.circle(screen, (255, 255, 255), (int(result.ball_x), int(result.ball_y)), BALL_RADIUS)
    pygame.display.flip()
    clock.tick(60)

    if result.game_over:
        print(f'Game over! Player {result.winner} wins.')
        running = False

pygame.quit()


Game over! Player 2 wins.
